# The Company Y Case Study

As part of the Supply Chain Management and Logistics Network Design course - Master's degree in Management Engineering - University of Bergamo

(c) Prof. Roberto Pinto

**This notebook contains all the data and functions required by the Company Y case study. 
Press shift + enter in each cell to run the commands, or click on Run in the command bar.**

In [ ]:
# Prepare the notebook
!git clone --branch projectwork_25_26 https://github.com/ropinotex/network_optimization.git
import sys
sys.path.insert(0,'/content/network_optimization')

!pip install -r /content/network_optimization/requirements.txt

Import functions and data

In [ ]:
# Data structures and utilities
from data_structures import show_geo_map
from netopt_utils import plot_map, load_data_from_spreadsheet

# Import the solver's user interface
from netopt_ui import netopt_ui

from netopt import netopt

In [ ]:
# Load data from an Excel speadsheet
# Running this cell, an Upload button should appear, enabling the selection of the source spreadsheet from your computer
# The software expect to get a spreadsheet in the same format as the one distributed. Therefore, do not modify the column's headers
# Once the file has been selected, the software should automatically load the data and display the first few lines.
warehouses, customers = load_data_from_spreadsheet()

**Check data**

Always check that the data is correctly loaded. To do this, just print the variables _customers_ and _warehouses_

In [ ]:
customers

# In the list that appears, the first number before the ":" is the index of the customer:
# the software would use this index to reference the customers in the solver and in the solution

In [ ]:
warehouses

# In the list that appears, the first number before the ":" is the index of the warehouse:
# the software would use this index to reference the customers in the solver 
# (for example, for the Force open or Force clsoed options)
# and in the solution

**Plot the data**

You can control the colors and shapes in the plot using the following parameters (also in the netopt function):
- warehouse_marker=shape of the warehouse icons. Allowed values are s=square, o=circle, *=star, ^=triangle, v=inverted triangle. Default is s 
- warehouse_markercolor=color of the warehouse icons. Allowed values are red, green, blue, black, yellow. Default is red
- warehouse_markersize=size of the warehouse icons. Default is 4
- warehouse_active_markersize=size of the warehouse icons representing active (open) warehouses. Default is 5
- customer_marker=shape of the customer icons. Default is o
- customer_markercolor=color of the customer icons. Default is blue
- customer_markersize=size of the customer icons. Default is 4

If you don't specify the above parameters about the figures, the default values will be assumed

In [ ]:
plot_map(customers=customers,
         warehouses=warehouses,
         warehouse_marker='s',
         warehouse_markercolor='red',
         warehouse_markersize=6,
         customer_marker='o',
         customer_markercolor='blue',
         customer_markersize=3)

A better map can be shown using the following command. 
However, this map allows only for the visualization of locations. 
To show the connections in the final solution you should use the plot_map command

In [ ]:
show_geo_map(warehouses=warehouses, customers=customers, zoom=6)

**Run the user interface**

The user interface allows you control the solver. The solver uses the data passed as arguments as _warehouses_ and _customers_.

- **Problem type**: The type of problem to be solved
- **# Warehouses**: Number of warehouses to activate (value of _p_ in the p-median problems)
- **Func. to minimize**: Minimize either the average waighted distance (AWD) or the total cost (only for p-median problems)
- **Coverage radius**: Defines the catchment area in a p-cover problem
- **Dist. Ranges**: Defines the cut points to measure the % of demand within given distances (i.e. [0, 100, 400] measures the % of demand within 100, 400, and beyond 400 km from any active warehouse)
- **Force single sourcing**: If active, each customer can be served by only on active warehouse
- **Force uncapacitated**: If active, neglects the capacities of the warehouse (only for p-median problems)
- **Ignore fixed cost**: If active, neglects the warehouses' fixed cost
- **Include unit handling cost**: If active, include the unit handling cost in the total cost
- **Force open**: Ensures that the listed warehouses are active in the final solution
- **Force closed**: Ensures that the listed warehouses are NOT active in the final solution
- **Force allocations**: Ensures that a given customer is allocated to a given warehouse (if the warehouse is active in the final solution) 
- **Mutually exclusive**: Prevents two warehouses to be simultaneously active at the same time in the final solution
- **Unit transportation cost**: Transportation cost per unit and km

**Warning:** The solver has a time limit of 120 seconds.

**Always check your data (warehouses and customers) before running the solver to be sure you are actually solving the right problem!**


In [ ]:
results = netopt_ui(warehouses=warehouses, customers=customers)
# Be sure to re-run this commando when you reload the data from the excel spreadsheet 
# or change the data in the _warehouses_ or _customers_ variables

In [ ]:
# To access all the results
results.result

## Some features of the solver ##

**Service levels**

By adding the parameter distance_ranges the functions returns the % of the demand within the passed distance ranges. 
For example, if distance_ranges = [0, 100, 400] the functions return the percentage of demand in the ranges [0, 100], (100, 400], (400, 99999]
where 99999 is used to represent a very long distance (i.e. infinite distance).

The parameter distance_ranges must be a list of increasing numbers. If you do not pass 0 as the first value it will be automatically added


**Force warehouses open or close**

It is possible to force warehouses to be open (for example, to force using the current warehouse) or closed (to avoid the selection of some candidates).

The warehouses are references through their id, and must be passed as list [] (even for a single value, that is to force closed the warehouse with id 1 you should pass force_closed=[1])

The id of the warehouses can be found in the 'warehouses' variable using the *show_data()* method.


**Mutually exclusive facilities**

In some cases, some facilities may be _mutually exclusive_, that is the presence of one facility excludes the activation of another one and viceversa.

For example, let's assume that facilities 1, 9 and 15 are mutually exclusive (either one of them or none of them can be selected). Similarly, warehouse 2 and 4 are mutually exclusive (but they are not exclusive with respect to 1, 9 and 15). This can be formulated by passing the parameter

_mutually_exclusive = [(1, 9, 15), (2, 4)]_

It is possible to set any number of mutually exclusive sets (each set is a tuple in a list). This constraint is useful when we have different alternatives (for example, different sizes) for the same facility.

**Remove the single-source constraint**

The model implicitly stipulates the single-source constraint (each customer is served by exactly one warehouse). When dealing with capacities, it may be useful to relax this constraint to better exploit the available capacity.



## Change the data ##

The simpler way to change data is to update the excel spreadsheet and reload the data

**It is also possible to run the solver programmatically.**

This is useful to automate the execution of different tests.


In [ ]:
# Import the solver
from netopt_compat import netopt

# Run the solver
results_prog=netopt(
    num_warehouses=3,
    warehouses=warehouses,
    customers=customers,
    distance_ranges=[],
    objective="UFLP",  # Accepted values: "p-median", "p-cover", "totalcover", "UFLP", "CFLP"
    objective_function="mincost",  # Objetive function for the p-median. Accepted values: "mindistance", "mincost"
    unit_transport_cost=0.2,
    mutually_exclusive=[],
    hide_inactive=False,
    force_single_sourcing=False,
    force_uncapacitated=False,
    ignore_fixed_cost=False,
    include_unit_handling_cost=True,
    force_open=[],
    force_closed=[],
    force_allocations=[],
    plot=True,
    plot_size=(8, 12),
    warehouse_marker="o",
    warehouse_markercolor="green",
    warehouse_markersize=6,
    customer_marker="s",
    customer_markercolor="blue",
    customer_markersize=6,
)

## From here on, it is up to you! ##